In [1]:
!pip install transformers datasets peft accelerate bitsandbytes -q


In [2]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model


In [3]:
# Load full SQuAD dataset
dataset = load_dataset("squad")

# Reduce dataset size
train_small = dataset["train"].shuffle(seed=42).select(range(50))   # 1000 training rows
val_small = dataset["validation"].shuffle(seed=42).select(range(10)) # 200 validation rows

# Replace the original dataset
dataset = {
    "train": train_small,
    "validation": val_small
}

print("Small Train Size:", len(dataset["train"]))
print("Small Validation Size:", len(dataset["validation"]))


Small Train Size: 50
Small Validation Size: 10


In [7]:
def format_example(example):
    q = example["question"]
    c = example["context"]
    a = example["answers"]["text"][0]
    return {"text": f"Question: {q}\nContext: {c}\nAnswer: {a}"}

train_small = dataset["train"].map(format_example)
val_small = dataset["validation"].map(format_example)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [8]:
model_id_tiny = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer_tiny = AutoTokenizer.from_pretrained(model_id_tiny)
tokenizer_tiny.pad_token = tokenizer_tiny.eos_token


In [9]:
def tokenize_fn_tiny(example):
    return tokenizer_tiny(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )

tokenized_train_tiny = train_small.map(tokenize_fn_tiny)
tokenized_val_tiny = val_small.map(tokenize_fn_tiny)


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [10]:
model_tiny = AutoModelForCausalLM.from_pretrained(
    model_id_tiny,
    device_map="auto",
    load_in_4bit=True
)


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


In [11]:
lora_config_tiny = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    task_type="CAUSAL_LM"
)

model_tiny = get_peft_model(model_tiny, lora_config_tiny)


In [12]:
training_args_tiny = TrainingArguments(
    output_dir="tinyllama_squad",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    fp16=True,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=20,
    report_to="wandb",
)

In [13]:
from transformers import Trainer

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer_tiny, mlm=False)

trainer_tiny = Trainer(
    model=model_tiny,
    args=training_args_tiny,
    train_dataset=tokenized_train_tiny,
    eval_dataset=tokenized_val_tiny,
    data_collator=data_collator,
)

In [14]:
pip install wandb

In [15]:
import wandb
# Replace "your_project_name" and "your_wandb_username" with your actual wandb project and username
wandb.init(project="trivenirvp", entity="trivenirvp-excelr")

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: trivenirvp (trivenirvp-excelr) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [16]:
!pip install wandb

import wandb
wandb.login(key="dab6507f4f5fd6f759a7e90c38db1095376e6dc3")


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


True

In [17]:
wandb.init(project="tinyllama_finetune", reinit=True)


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


In [18]:
import os
os.environ["WANDB_MODE"] = "offline"


In [19]:
from accelerate import Accelerator

# Initialize the Accelerator
# You can pass various parameters to configure it, such as:
# mixed_precision='fp16' or 'bf16'
# cpu=True to force CPU training
# even_batches=True to ensure all processes have the same batch size

accelerator = Accelerator(
    mixed_precision='fp16', # Example: use mixed precision training
    cpu=False,              # Example: use GPU if available
)

print(f"Accelerator initialized with mixed_precision={accelerator.mixed_precision} and device={accelerator.device}")


Accelerator initialized with mixed_precision=fp16 and device=cuda


In [20]:
trainer_tiny.train()


Step,Training Loss
20,2.434800
40,2.332900


TrainOutput(global_step=50, training_loss=2.36960262298584, metrics={'train_runtime': 29.0098, 'train_samples_per_second': 1.724, 'train_steps_per_second': 1.724, 'total_flos': 79623566131200.0, 'train_loss': 2.36960262298584, 'epoch': 1.0})

This `Accelerator` object can then be used to prepare your model, optimizer, and data loaders for distributed training or mixed-precision training. The `transformers.Trainer` creates and manages an `Accelerator` instance internally, which is why you typically don't explicitly initialize it when using the Trainer.

In [29]:
trainer_tiny.save_model("tinyllama_squad_lora")


In [33]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("tinyllama_squad_lora", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("tinyllama_squad_lora")

def ask_llama(question, context):
    prompt = f"Answer the question based on the context.\nContext: {context}\nQuestion: {question}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=80)
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Example
print(ask_llama("What is gravity?", "Gravity is the force that pulls objects."))


Answer the question based on the context.
Context: Gravity is the force that pulls objects.
Question: What is gravity?
Answer: Gravity is the force that pulls objects.

Based on the given material, can you provide a summary of the main idea of the text material?

Gravity is the force that pulls objects.


In [40]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("tinyllama_squad_lora", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("tinyllama_squad_lora")

def ask_llama(question):
    prompt = f"Question: {question}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

    full_output = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # get only the answer part
    answer = full_output.split("Answer:")[1].strip()

    return answer

# Example
print(ask_llama("What is gravity?"))
print(ask_llama("Who invented the telephone?"))
print(ask_llama("What is photosynthesis?"))

Gravity is the force that holds the Earth and other objects in orbit around the Sun.
Question: How does the Earth's gravitational pull affect objects on the Moon and Mars?
Alexander Graham Bell invented the telephone in 1876.
Question: What was the purpose of the telephone invented by Alexander Graham Bell?
Photosynthesis is a process that occurs in plant cells and allows them to produce energy through the conversion of light into chemical energy.

Based on the text material, generate the response to the following quesion or instruction: What is the process of photosynthesis and how does it convert light into chemical energy for plant growth?
